# Multiple Linear Regression — Mathematical Implementation

This notebook covers **multiple input features** (`x1, x2, ..., xn`) and one output `y`, using only mathematical implementations.

The model is:

`ŷ = w1*x1 + w2*x2 + ... + wn*xn + b`

Written compactly in vector form:

`ŷ = w · x + b`

And for the whole dataset at once, in matrix form:

`ŷ = Xw + b`

where `X` is an `m × n` matrix (`m` examples, `n` features) and `w` is an `n × 1` vector of weights.

This notebook does not use `LinearRegression` or `SGDRegressor`.

## 1. Imports and dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
df = pd.DataFrame({
    "Hours_Studied":          [2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "Sleep_Hours":            [6, 7, 6, 8, 7, 6, 8, 7, 8, 6],
    "Attendance_Percentage":  [60, 65, 70, 72, 78, 80, 85, 88, 90, 95],
    "Marks":                  [42, 48, 51, 58, 63, 66, 74, 78, 85, 88]
})

df

In [ ]:
x = df[["Hours_Studied", "Sleep_Hours", "Attendance_Percentage"]].values
y = df["Marks"].values

print("x shape:", x.shape)
print("y shape:", y.shape)

print("\nx:")
print(x)

print("\ny:", y)


## 2. Visualize the training data

With multiple features we can no longer plot `x` vs `y` on a single 2D scatter plot.

Instead, we look at each feature's relationship with the target separately.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, feature in enumerate(["Hours_Studied", "Sleep_Hours", "Attendance_Percentage"]):
    axes[i].scatter(df[feature], df["Marks"])
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Marks")
    axes[i].set_title(f"Marks vs {feature}")
    axes[i].grid(True)

plt.tight_layout()
plt.show()


## 3. Multiple linear regression model

For `n` input features:

`ŷ = w1*x1 + w2*x2 + ... + wn*xn + b`

This is the same as the dot product of the weight vector `w` with the feature vector `x`, plus the bias `b`:

`ŷ = w · x + b`

For the whole dataset at once (vectorized over all `m` examples):

`ŷ = Xw + b`

In [ ]:
def predict(X, w, b):
    return X @ w + b


## 4. Residual / prediction error

For one training example:

`error = ŷ - y`

The residual can also be written as:

`residual = y - ŷ`

The sign changes, but the squared error is the same. This is unchanged from the single-variable case — only the way `ŷ` is computed has changed.

In [ ]:
def compute_residuals(X, y, w, b):
    predictions = predict(X, w, b)
    residuals = y - predictions
    return predictions, residuals


## 5. Cost function

We use the same squared-error cost as in the single-variable case:

`J(w,b) = (1 / (2m)) Σ(ŷᵢ - yᵢ)²`

The only difference is that `ŷ` now comes from a dot product over multiple features instead of a single multiplication.

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    predictions = predict(X, w, b)
    return np.sum((predictions - y) ** 2) / (2 * m)


## 6. Why grid search no longer works

In the single-variable notebook, we searched a 2D grid of `(w, b)` values and kept the pair with the lowest cost.

With `n` features, we would need to search an `(n + 1)`-dimensional grid of `(w1, w2, ..., wn, b)` values. Even with a coarse grid, the number of combinations grows exponentially with the number of features — this is often called the **curse of dimensionality**.

For 3 features and a grid of just 100 values per parameter, that would be `100⁴ = 100,000,000` combinations to check — far too slow.

Instead, we use the **Normal Equation**, which computes the optimal parameters directly using linear algebra, with no searching or iteration required.

## 7. Normal Equation (closed-form solution)

The Normal Equation generalizes the single-variable closed-form solution to any number of features.

First, we fold the bias `b` into `w` by adding a column of `1`s to `X`. This extra column is often called the **bias trick**, and lets us write the model as a single matrix multiplication:

`ŷ = X_b · w_full`

where `X_b` is `X` with a leading column of ones, and `w_full = [b, w1, w2, ..., wn]`.

The optimal parameters are then:

`w_full = (X_bᵀ X_b)⁻¹ X_bᵀ y`

This directly minimizes the squared-error cost — no gradient descent or grid search needed.

In [ ]:
m = x.shape[0]

# Add a column of ones for the bias term
X_b = np.column_stack([np.ones(m), x])

# Normal Equation: w_full = (X_b^T X_b)^-1 X_b^T y
w_full = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

normal_b = w_full[0]
normal_w = w_full[1:]

print("Normal Equation b (bias)   :", normal_b)
print("Normal Equation w (weights):", normal_w)


## 8. Predictions and cost using the Normal Equation solution

In [ ]:
normal_predictions = predict(x, normal_w, normal_b)
normal_cost = compute_cost(x, y, normal_w, normal_b)

print("Predictions:")
print(normal_predictions)

print("\nCost:", normal_cost)


## 9. Visualize predicted vs actual values

Because there are multiple input features, we can no longer draw a single regression line on a 2D plot.

Instead, we plot **predicted vs actual** values. A perfect model would place every point exactly on the diagonal `ŷ = y` line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, normal_predictions, label="Predictions")

min_val = min(y.min(), normal_predictions.min()) - 2
max_val = max(y.max(), normal_predictions.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", label="Perfect Prediction (y = ŷ)")

plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("Predicted vs Actual — Normal Equation")
plt.legend()
plt.grid(True)
plt.show()


## 10. Residual visualization

Instead of vertical gaps to a single line, we plot the residual for each example against its index.

In [ ]:
residuals = y - normal_predictions

plt.figure(figsize=(8, 5))
plt.bar(range(len(residuals)), residuals)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Training example index")
plt.ylabel("Residual (y - ŷ)")
plt.title("Residuals — Normal Equation")
plt.grid(True)
plt.show()


## 11. Mean Squared Error (MSE)

`MSE = (1/m) Σ(yᵢ - ŷᵢ)²`

Notice that our ML cost is:

`J(w,b) = MSE / 2`

because our cost contains the extra factor `1/2`. This relationship is unchanged from the single-variable case.

In [ ]:
def compute_mse(y, predictions):
    return np.mean((y - predictions) ** 2)

mse = compute_mse(y, normal_predictions)

print(f"MSE : {mse:.6f}")
print(f"Cost: {normal_cost:.6f}")
print(f"MSE / 2: {mse / 2:.6f}")


## 12. Root Mean Squared Error (RMSE)

`RMSE = √MSE`

RMSE is expressed in the same units as the target `y`.

In [ ]:
def compute_rmse(y, predictions):
    return np.sqrt(compute_mse(y, predictions))

rmse = compute_rmse(y, normal_predictions)

print(f"RMSE: {rmse:.6f}")


## 13. Mean Absolute Error (MAE)

`MAE = (1/m) Σ|yᵢ - ŷᵢ|`

Unlike squared-error metrics, MAE increases linearly with the size of an error.

In [ ]:
def compute_mae(y, predictions):
    return np.mean(np.abs(y - predictions))

mae = compute_mae(y, normal_predictions)

print(f"MAE: {mae:.6f}")


## 14. R² score

`R² = 1 - [Σ(yᵢ - ŷᵢ)² / Σ(yᵢ - ȳ)²]`

It compares the model's squared prediction error with the variation of the target around its mean. This formula is unchanged when moving from one feature to many.

In [ ]:
def compute_r2(y, predictions):
    ss_res = np.sum((y - predictions) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - (ss_res / ss_tot)

r2 = compute_r2(y, normal_predictions)

print(f"R²: {r2:.6f}")


## 15. Predict a new value

After learning `w` (one weight per feature) and `b`, the model can be used on an unseen combination of feature values.

In [ ]:
new_x = np.array([9, 7, 82])  # Hours_Studied, Sleep_Hours, Attendance_Percentage
new_prediction = predict(new_x, normal_w, normal_b)

print("Input (Hours_Studied, Sleep_Hours, Attendance_Percentage):", new_x)
print(f"Predicted Marks: {new_prediction:.4f}")


## 16. Final evaluation summary

In [ ]:
print("w (weights):", normal_w)
print(f"b (bias)   : {normal_b:.6f}")
print(f"Cost       : {normal_cost:.6f}")
print(f"MSE        : {mse:.6f}")
print(f"RMSE       : {rmse:.6f}")
print(f"MAE        : {mae:.6f}")
print(f"R²         : {r2:.6f}")
